# Phase 2b (v2) — KD-LoRA Training, Fixed for 40GB + Completion-Only Loss

**Two fixes vs v1:**
1. **Completion-only loss.** Student only computes loss on the *summary* tokens, not the article. This was the main reason loss wasn't decreasing — most of the prior gradient signal was wasted trying to predict arbitrary article tokens.
2. **Left-truncate articles.** If an article is too long, we truncate from the *start* so the summary at the end is always preserved. (Previously, long articles got truncated from the right, which chopped off the summary the student was supposed to learn.)

**VRAM fixes for 40GB:**
- `max_seq_length` reduced 2048 → 1536
- `per_device_batch_size` 4 → 2, `gradient_accumulation_steps` 4 → 8 (effective batch 16, same as before)
- Gradient checkpointing explicitly enabled with PEFT-safe wiring
- `optim='adamw_torch_fused'` for ~10% memory savings

**Other tweaks:**
- Learning rate 2e-4 → 3e-4 (more typical for small-model LoRA SFT)
- Logging every 10 steps so you can see loss moving sooner

In [1]:
!pip install -q transformers==4.46.0 datasets==2.21.0 accelerate==1.0.1 peft==0.13.2 trl==0.11.4 sentencepiece

In [1]:
import torch, json
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from transformers import EarlyStoppingCallback

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

CONFIG = {
    'student_model': 'Qwen/Qwen2.5-0.5B',
    'teacher_data': 'teacher_generations.jsonl',
    'output_dir': './model_outputs/student_kd_lora',   # different from SFT
    'target_field': 'teacher_summary',                       # KD target
    'num_epochs': 3,
    'per_device_batch_size': 4,
    'gradient_accumulation_steps': 4,
    'learning_rate': 2e-4,
    'warmup_ratio': 0.03,
    'max_seq_length': 1536,
    'max_article_chars': 6000,
    'lora_r': 32,
    'lora_alpha': 64,
    'lora_dropout': 0.1,
    'weight_decay': 0.01,
    'eval_split_size': 500,
    'seed': 42,
}
Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, indent=2))

GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
{
  "student_model": "Qwen/Qwen2.5-0.5B",
  "teacher_data": "teacher_generations.jsonl",
  "output_dir": "./model_outputs/student_kd_lora",
  "target_field": "teacher_summary",
  "num_epochs": 3,
  "per_device_batch_size": 4,
  "gradient_accumulation_steps": 4,
  "learning_rate": 0.0002,
  "warmup_ratio": 0.03,
  "max_seq_length": 1536,
  "max_article_chars": 6000,
  "lora_r": 32,
  "lora_alpha": 64,
  "lora_dropout": 0.1,
  "weight_decay": 0.01,
  "eval_split_size": 500,
  "seed": 42
}


## 1. Load teacher generations

In [2]:
records = []
with open(CONFIG['teacher_data']) as f:
    for line in f:
        records.append(json.loads(line))
print(f'Loaded {len(records)} records')
print('Sample target ({}): {}'.format(CONFIG['target_field'], records[0][CONFIG['target_field']]))

Loaded 50000 records
Sample target (teacher_summary): Three members of the same family died from carbon monoxide poisoning in a static caravan in Cornwall; investigators stated the victims would have been unconscious within minutes due to the lack of a working detector.


## 2. Format examples — left-truncate long articles, preserve summary

The key insight: tokenizer truncation defaults to right-truncation, which kills the assistant response. We pre-truncate the article from the *left* (keeping the most recent part), then format the chat template. Whatever tokenizer truncation happens after that, the assistant response is at the very end and protected by `max_seq_length`.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG['student_model'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'  # for training (causal LM)
tokenizer.truncation_side = 'left'  # if we still need to truncate, drop from start

SYSTEM_PROMPT = (
    'You are a concise news summarizer. Write a short summary of the article in 2-3 sentences. '
    'Output only the summary itself, with no preamble, headers, or commentary.'
)
USER_TEMPLATE = 'Article:\n{article}\n\nSummary:'

def format_example(rec) -> dict:
    # char-level left-truncate first
    article = rec['article']
    if len(article) > CONFIG['max_article_chars']:
        article = article[-CONFIG['max_article_chars']:]
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': USER_TEMPLATE.format(article=article)},
        {'role': 'assistant', 'content': rec[CONFIG['target_field']]},
    ]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False)}

formatted = [format_example(r) for r in records]
full_ds = Dataset.from_list(formatted)
split = full_ds.train_test_split(test_size=CONFIG['eval_split_size'], seed=CONFIG['seed'])
train_ds, eval_ds = split['train'], split['test']
print(f'Train: {len(train_ds)}  Eval: {len(eval_ds)}')
print('\n--- Sample formatted text (end of sequence) ---')
print(formatted[0]['text'][-500:])

Train: 49500  Eval: 500

--- Sample formatted text (end of sequence) ---
tle Jack Russell, it is so sad . what has happened, I understand the dog went with them. 'They . will be sorely missed and I think everyone is just in shock at the . moment, I would like to send my condolences to the Cook family.'

Summary:<|im_end|>
<|im_start|>assistant
Three members of the same family died from carbon monoxide poisoning in a static caravan in Cornwall; investigators stated the victims would have been unconscious within minutes due to the lack of a working detector.<|im_end|>



## 3. Completion-only data collator

This is the critical fix. The collator masks out everything before the assistant turn marker so cross-entropy loss is computed ONLY on the summary tokens. Without this, the model wastes most of its gradient capacity trying to predict the article — which is unlearnable noise from a small model's perspective — and loss appears stuck.

In [4]:
# Qwen chat template ends each turn with <|im_end|>\n and starts assistant with <|im_start|>assistant\n
RESPONSE_TEMPLATE = '<|im_start|>assistant\n'

# Verify the template tokenizes consistently (TRL needs it to find a match in every example)
response_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
print(f'Response template token ids: {response_ids}')
print(f'Decoded back: {tokenizer.decode(response_ids)!r}')

# Sanity: confirm template appears in formatted text
assert RESPONSE_TEMPLATE in formatted[0]['text'], 'response template not found — chat template mismatch'
print('✓ Response template found in formatted examples')

collator = DataCollatorForCompletionOnlyLM(
    response_template=RESPONSE_TEMPLATE,
    tokenizer=tokenizer,
)

Response template token ids: [151644, 77091, 198]
Decoded back: '<|im_start|>assistant\n'
✓ Response template found in formatted examples


## 4. Load student + LoRA, with PEFT-safe gradient checkpointing

In [5]:
model = AutoModelForCausalLM.from_pretrained(
    CONFIG['student_model'], torch_dtype=torch.bfloat16, device_map=DEVICE)
model.config.use_cache = False

# CRITICAL when combining gradient checkpointing with PEFT/LoRA:
# without this, gradients can fail to flow into the LoRA adapters
model.gradient_checkpointing_enable()
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()

lora_config = LoraConfig(
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 17,596,416 || all params: 511,629,184 || trainable%: 3.4393


## 5. Train

In [6]:
sft_config = SFTConfig(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['per_device_batch_size'],
    per_device_eval_batch_size=CONFIG['per_device_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    gradient_checkpointing=False,
    gradient_checkpointing_kwargs={'use_reentrant': False},  # safer with PEFT
    learning_rate=CONFIG['learning_rate'],
    lr_scheduler_type='cosine',
    warmup_ratio=CONFIG['warmup_ratio'],
    bf16=True,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    optim='adamw_torch_fused',
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=3,
    max_seq_length=CONFIG['max_seq_length'],
    dataset_text_field='text',
    packing=False,  # MUST be False with DataCollatorForCompletionOnlyLM
    report_to='none',
    seed=CONFIG['seed'],
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    data_collator=collator,   # <-- this is what enables completion-only loss
)

# Quick gradient-flow sanity check on one batch before committing to the full run
print('Running 1-step sanity check (should show non-zero loss)...')
batch = next(iter(trainer.get_train_dataloader()))
batch = {k: v.to(DEVICE) for k, v in batch.items()}
out = model(**batch)
print(f'  Initial loss: {out.loss.item():.4f}')
assert out.loss.item() > 0 and not torch.isnan(out.loss), 'Loss is zero or NaN — something is wrong'
# also confirm masked label positions exist (i.e. completion-only is doing its job)
n_unmasked = (batch['labels'] != -100).sum().item()
n_total = batch['labels'].numel()
print(f'  Unmasked label tokens: {n_unmasked} / {n_total} ({100*n_unmasked/n_total:.1f}%)')
print(f'  → If this is ~5-15%, completion-only masking is working correctly.')
del out, batch; torch.cuda.empty_cache()

Map:   0%|          | 0/49500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


Running 1-step sanity check (should show non-zero loss)...
  Initial loss: 1.7269
  Unmasked label tokens: 258 / 4568 (5.6%)
  → If this is ~5-15%, completion-only masking is working correctly.


In [7]:


# Then restart and rerun OR force-reload:
import torch
import functools

# Save original
_original_torch_load = torch.load

# Patch to default weights_only=False
@functools.wraps(_original_torch_load)
def _patched_torch_load(*args, **kwargs):
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)

torch.load = _patched_torch_load

# Now resume
trainer.train(resume_from_checkpoint='./model_outputs/student_kd_lora/checkpoint-3600')

# Optional: restore after
torch.load = _original_torch_load

trainer.save_model(CONFIG['output_dir'])

log_path = Path(CONFIG['output_dir']) / 'training_log.json'
with open(log_path, 'w') as f:
    json.dump(trainer.state.log_history, f, indent=2)
print(f'\nSaved adapter + log to {CONFIG["output_dir"]}')

Step,Training Loss,Validation Loss
3800,1.034000,1.307760
4000,1.036600,1.298261
4200,1.071000,1.293602



Saved adapter + log to ./model_outputs/student_kd_lora


In [ ]:
from huggingface_hub import login, HfApi

# Paste your HF write token here OR set HF_TOKEN env var and call login() with no args
login(token='hf_TOEKN')

# Fill in your HF username
HF_USERNAME = 'Harsha901'
REPO_NAME = 'qwen2.5-0.5b-kd-lora-cnndm-50k'   # change to '-sft-lora-cnndm' when you push the SFT model
REPO_ID = f'{HF_USERNAME}/{REPO_NAME}'

# Push the LoRA adapter and tokenizer
trainer.model.push_to_hub(
    REPO_ID,
    commit_message='KD-LoRA student on CNN/DailyMail, best val_loss checkpoint',
    private=True,   # set False to make public
)
tokenizer.push_to_hub(REPO_ID, private=True)

# Add a README with reproducibility info
readme = f"""---
base_model: Qwen/Qwen2.5-0.5B
library_name: peft
tags:
- lora
- summarization
- distillation
- cnn_dailymail
---

# {REPO_NAME}

LoRA adapter for `Qwen/Qwen2.5-0.5B` fine-tuned via **sequence-level knowledge distillation** from `Qwen/Qwen2.5-7B-Instruct` teacher generations on the CNN/DailyMail summarization dataset.

## Training config
- LoRA: r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}, dropout={CONFIG['lora_dropout']}
- Target modules: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
- Effective batch size: {CONFIG['per_device_batch_size'] * CONFIG['gradient_accumulation_steps']}
- Learning rate: {CONFIG['learning_rate']}, cosine schedule, {CONFIG['warmup_ratio']:.0%} warmup
- Weight decay: {CONFIG['weight_decay']}
- Max sequence length: {CONFIG['max_seq_length']}
- Trained for up to {CONFIG['num_epochs']} epochs with early stopping on `eval_loss`
- Best checkpoint loaded at end of training

## Usage
```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B', torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, '{REPO_ID}')
tokenizer = AutoTokenizer.from_pretrained('{REPO_ID}')
```
"""
api = HfApi()
api.upload_file(
    path_or_fileobj=readme.encode(),
    path_in_repo='README.md',
    repo_id=REPO_ID,
    commit_message='Add README',
)

print(f'\n✓ Pushed to: https://huggingface.co/{REPO_ID}')

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  57%|#####6    | 39.9MB / 70.4MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpvjaexa1u/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            


✓ Pushed to: https://huggingface.co/Harsha901/qwen2.5-0.5b-kd-lora-cnndm-50k


## What to expect now

**Initial loss:** should start somewhere around 2.0-3.5 (cross-entropy on summary tokens for an untrained-on-task student).

**After 50-100 steps:** should drop visibly, maybe to 1.5-2.0.

**By end of epoch 1:** typically 0.8-1.3 range.

**If loss is still flat after 50 steps:** check the sanity-check output above. If unmasked label percentage is near 0%, the response template isn't being found — print one tokenized example and inspect. If unmasked percentage looks fine but loss is flat, the LoRA isn't getting gradients — likely a gradient checkpointing wiring issue.